# P4. 테스트 DB 적재 (공동 Aura, 접두사·테스트 라벨 방식)

## 이 노트북이 하는 일
- 블록 8개(`blocks.jsonl`)를 Neo4j에 넣고, 재실행해도 중복이 없는지 확인한 뒤, 테스트 데이터를 **삭제**합니다.
- Aura는 **팀 공동 DB**입니다. 그래서 다음 규칙을 지킵니다.
  - 노드 라벨은 테스트 전용 `PLTest` 하나만 씁니다. `Chunk` 라벨은 쓰지 않습니다. 팀원 쿼리에 섞이지 않게 하기 위해서입니다.
  - 노드 `id`는 `test_` 접두사를 붙입니다.
  - 삭제는 `PLTest` 라벨이고 `test_` 접두사인 노드만 대상으로 합니다.
- 실제 적재(공통 ID)는 최종 결과가 나온 뒤에만 합니다. 이 노트북에서는 하지 않습니다.

## 1. 연결과 전후 노드 수 확인
삭제 전후의 전체 노드 수를 비교해서, 다른 데이터를 건드리지 않았는지 확인합니다.

In [1]:
from rag.prevention_legal.neo4j_conn import use_windows_trust, connect

# Windows 인증서 저장소를 쓰도록 설정한다 (검증은 그대로 켜 둔다)
use_windows_trust()
driver = connect()

TEST_LABEL = "PLTest"   # 테스트 전용 라벨
PREFIX = "test_"        # 노드 id 접두사

def count_all():
    # 전체 노드 수 (다른 데이터가 있는지 보는 용도)
    with driver.session() as s:
        return s.run("MATCH (n) RETURN count(n) AS c").single()["c"]

def count_test():
    # 테스트 라벨 노드 수
    with driver.session() as s:
        return s.run(f"MATCH (n:{TEST_LABEL}) RETURN count(n) AS c").single()["c"]

print("시작 시 전체 노드:", count_all(), "| 테스트 노드:", count_test())

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `PLTest` does not exist in database `9df2d804`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:PLTest) RETURN count(n) AS c'


시작 시 전체 노드: 0 | 테스트 노드: 0


## 2. 블록 적재 (MERGE 로 넣는다)
MERGE 는 같은 id 가 있으면 새로 만들지 않고 기존 노드를 쓴다. 그래서 두 번 실행해도 개수가 늘지 않는다.

In [2]:
import json

# 블록 파일을 읽는다 (노트북 위치 기준 상대 경로)
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
blocks = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]

# 테스트 노드를 만든다. 같은 id 가 있으면 값만 갱신한다.
CYPHER_LOAD = f"""
MERGE (c:{TEST_LABEL} {{id: $id}})
SET c.document_id = $document_id,
    c.pdf_page = $pdf_page,
    c.section = $section,
    c.text = $text,
    c.is_test = true
"""

def load_blocks():
    with driver.session() as s:
        for b in blocks:
            s.run(CYPHER_LOAD, id=PREFIX + b["chunk_id"], document_id=b["document_id"],
                  pdf_page=b["pdf_page"], section=b.get("section") or "", text=b["text"])

load_blocks()
print("1차 적재 후 테스트 노드:", count_test(), "(블록 수:", len(blocks), ")")

1차 적재 후 테스트 노드: 8 (블록 수: 8 )


## 3. 재실행 중복 확인
같은 블록을 한 번 더 넣었을 때 개수가 그대로여야 합니다. 늘면 중복 적재입니다.

In [3]:
load_blocks()
after_second = count_test()
print("2차 적재 후 테스트 노드:", after_second)
assert after_second == len(blocks), "중복 적재가 생겼습니다. 중단하고 원인을 확인하세요."
print("재실행 중복 없음 ✅")

2차 적재 후 테스트 노드: 8
재실행 중복 없음 ✅


## 4. 질의 확인 (읽기만)
적재된 노드가 실제로 찾아지는지 한 건만 봅니다.

In [4]:
with driver.session() as s:
    rows = s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p RETURN c.id AS id, c.pdf_page AS page LIMIT 3", p=PREFIX).data()
for r in rows:
    print(r["id"], "| 쪽", r["page"])

test_kdca_measles_2026:p0093:b01 | 쪽 93
test_kdca_measles_2026:p0093:b02 | 쪽 93
test_kdca_measles_2026:p0093:b03 | 쪽 93


## 5. 테스트 데이터 삭제
접두사와 테스트 라벨이 **둘 다** 붙은 노드만 지운다. 삭제 후 테스트 노드 0개, 전체 노드 수가 시작 값과 같은지 확인한다.

In [5]:
with driver.session() as s:
    s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p DETACH DELETE c", p=PREFIX)

left_test = count_test()
print("삭제 후 테스트 노드:", left_test)
assert left_test == 0, "테스트 노드가 남았습니다."
print("삭제 후 전체 노드:", count_all())
driver.close()

삭제 후 테스트 노드: 0
삭제 후 전체 노드: 0
